In [ ]:

import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

# Load and preprocess data
df = pd.read_csv("LP IV Updated Datasets/AutoEncoder/CreditCard/creditcard.csv")
train, test = train_test_split(df, test_size=0.2, stratify=df.Class)

scaler = StandardScaler()
x_train = scaler.fit_transform(train[train.Class == 0].drop(columns="Class"))
x_test = scaler.transform(test.drop(columns="Class"))

# Create autoencoder
model = Sequential([
    Input(shape=(30,)),
    Dense(8, activation='relu'),
    Dense(30)
])

# Compile
model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

# Train
model.fit(x_train, x_train, epochs=10, batch_size=128)

# Calculate reconstruction error and threshold
error = np.mean((x_train - model.predict(x_train)) ** 2, axis=1)
threshold = np.percentile(error, 99.5)

error = np.mean((x_test - model.predict(x_test)) ** 2, axis=1)
predictions = error > threshold

# Evaluate fraud detection
print("Fraud Detected:", predictions.sum())
print(classification_report(test.Class, predictions, zero_division=0))
